In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import pytorch_lightning as pl
from torch.distributions import Bernoulli
import xarray as xr
import xrft
import numpy as np
import matplotlib.pyplot as plt
import src.data
import src.utils
from scipy.stats import multivariate_normal
import pandas as pd
import functools as ft
from collections import namedtuple
from IPython.display import Markdown, display
from omegaconf import OmegaConf
import yaml
import inspect
import hydra
from torchviz import make_dot
import os
from matplotlib.gridspec import GridSpec
from matplotlib.animation import FuncAnimation, PillowWriter
import matplotlib.colors as colors
from matplotlib.colors import LogNorm 
from matplotlib.animation import FuncAnimation
import matplotlib.pyplot as plt
from PIL import Image
import kornia.filters as kfilts
from torch.cuda.amp import autocast, GradScaler
from scipy.ndimage import gaussian_filter
from scipy.ndimage import binary_dilation
from skimage.morphology import disk, dilation
from sklearn.linear_model import LinearRegression

device = torch.device("cuda:6" if torch.cuda.is_available() else "cpu")
#device = "cpu"

In [3]:
tgt_ecs = xr.open_dataset('/Odyssey/public/natl60/celerity/NATL60GULF-CJM165_cutoff_freq_regrid_0_1000m.nc')

In [7]:
tgt_ecs.ecs.values

array([[[ 0.        ,  0.        ,  0.        , ...,  0.        ,
          0.        ,  0.        ],
        [ 0.        ,  0.        ,  0.        , ...,  0.        ,
          0.        ,  0.        ],
        [ 0.        ,  0.        ,  0.        , ...,  0.        ,
          0.        ,  0.        ],
        ...,
        [ 2.79421042,  0.        ,  0.        , ..., 48.33723778,
         48.33723778, 48.33723778],
        [ 2.79421042,  0.        ,  0.        , ..., 52.48403405,
         52.48403405, 52.48403405],
        [ 0.        ,  0.        ,  0.        , ..., 52.48403405,
         52.48403405, 52.48403405]],

       [[ 0.        ,  0.        ,  0.        , ...,  0.        ,
          0.        ,  0.        ],
        [ 0.        ,  0.        ,  0.        , ...,  0.        ,
          0.        ,  0.        ],
        [ 0.        ,  0.        ,  0.        , ...,  0.        ,
          0.        ,  0.        ],
        ...,
        [ 1.55879158,  0.        ,  0.        , ..., 4

In [5]:
norm_per_day = np.linalg.norm(tgt_ecs.ecs, axis=(1, 2))
norm_data_array = xr.DataArray(norm_per_day, coords=[tgt_ecs.time], dims=['time'])

In [6]:
norm_data_array

<xarray.DataArray (time: 365)>
array([ 9062.35458749,  7849.49748188,  8651.61992553,  9805.78519235,
       10370.80662444,  8384.4542732 ,  6821.31584099,  8873.50127012,
       11127.85733936, 10679.08837678, 11392.88567972, 12016.25836015,
       12634.96909714, 13345.40479348, 13711.65961866, 13731.86437484,
       13677.1984959 , 11974.46361514, 13086.46416304, 13552.54246985,
       13979.77719107, 14287.4339681 , 14792.69286739, 15305.73190335,
       15699.33861546, 16253.10852004, 16763.4759345 , 17043.50379915,
       16701.82958998, 16287.79236724, 16481.726324  , 16597.13887107,
       16439.52523913, 17497.00444936, 17945.43185669, 17380.3685805 ,
       17763.38108993, 18465.51744335, 17955.14873746, 18287.38487188,
       18780.27312533, 19282.62029306, 19544.01275826, 19728.8895935 ,
       19553.884195  , 19363.6414497 , 17275.97652426, 19120.99024482,
       19861.56627705, 20239.97280269, 20525.67364508, 20735.38892444,
       20592.03096937, 21165.97750982, 20993.88596539, 21455.18219974,
       22032.0700783 , 22001.50787612, 22434.44772981, 22632.47282624,
       23199.61215937, 23653.96591179, 24012.03293698, 24199.61395261,
       24177.42272821, 24262.93408604, 24483.45492964, 24906.97546304,
       25255.5124055 , 25434.08004136, 25265.45584711, 25518.06264199,
       25600.93097334, 25744.43680633, 25895.0044786 , 26149.31400141,
       26533.39210828, 26743.03381511, 26939.61480414, 27005.29402703,
...
         624.30028801,  1159.01929337,  1046.45521419,   406.0604258 ,
        1206.63940206,   948.60965522,  1245.72152831,  1834.16354087,
        2385.3295382 ,  1935.9912061 ,  2739.35122435,  3010.4404925 ,
        1179.03016698,   612.64078106,  1869.01990146,   782.24111396,
        1499.53907856,  3258.90366469,  2546.81959971,  3716.05496382,
        3901.1701315 ,  3204.4062588 ,  3148.46525886,  1700.56149133,
        1358.04505126,  3122.90669461,  2096.32531936,  1240.03350005,
        1631.4413274 ,  3538.69014647,  3996.20018506,  1997.28323464,
        2527.88281757,  2399.34518506,  4126.30284794,  3900.91463711,
        2939.36816579,  2504.20911434,  3269.4083242 ,  5273.75446524,
        6541.77431537,  5800.12177415,  3797.48460745,  5013.25730265,
        6338.40863859,  7194.0741267 ,  3661.92478823,  2582.30633707,
        1841.00432911,  3548.32348588,  5870.40999896,  5933.28401108,
        5884.08318512,  3231.58850165,  6601.73997852,  6084.96593794,
        4422.5027925 ,  3426.37582043,  3177.5582635 ,  6499.03153811,
        6723.82173926,  6757.97929219,  5124.76937839,  6630.41514865,
        8393.96189154,  9197.25254633,  9182.52555329,  9221.38491455,
        8998.20356197,  9752.51626948,  9855.60805471,  9886.35096082,
       10587.06601836, 11361.54804875, 11046.4710881 , 10848.20863766,
       10951.42046375])
Coordinates:
  * time     (time) datetime64[ns] 2012-10-01T12:00:00 ... 2013-09-30T12:00:00

In [ ]:
class CNN_normalization(nn.Module):
    def __init__(self):
        super(CNN_normalization, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(32 * 120 * 120, 128) # Adjust the input size based on the output of the previous layer
        self.fc2 = nn.Linear(128, 1)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = x.view(-1, 32 * 120 * 120)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x